# Alias-config reconstruction — FMDV

Đưa tool một **FMDV mới tinh** (chỉ tên gene của reference, chưa có alias map): tầng gợi ý dựa-trên-tọa-độ
dựng lại được config curated tới đâu?

**Ba bước, gọi ĐÚNG hàm pipeline** (`ui/stages/bootstrap_alias.py`), không reimplement:
`build_coordinate_supported_alias_suggestions` → `review_uncertain_alias_suggestions` (LLM) →
`apply_approved_alias_suggestions` (dựng `config_temp`).

> **Đo hai chiều, per-canonical:**
> - **PRECISION** (duyệt config_temp): tool bỏ tên gì dưới canon X → truth có đồng ý thuộc X không?
>   Bắt `wrong_gene` (map sai), `false_save` (lưu nhầm rác).
> - **RECALL** (duyệt config_truth, **gate theo corpus**): alias thật truth có dưới X **mà thực sự xuất
>   hiện trong 100 record** → config_temp có giữ ở X không? Bắt `missed` (kiểu bug sM: alias thật bị loại).
>
> **Gate corpus (quan trọng):** alias truth **không** xuất hiện trong 100 record thì tool không có gì để
> gợi ý → **park, không tính accuracy**. Nếu không gate sẽ phạt oan tool.
>
> **Bước người không tự động được:** harness dừng ở khuyến nghị + chính sách auto-approve khai báo rõ
> (chấp nhận mọi dòng tool đánh `save`). Số là **cận trên**. Không API key → LLM mock (in rõ chế độ).

## Setup

In [1]:
from pathlib import Path
import os, sys
import pandas as pd

ROOT = Path.cwd()
for c in [ROOT, *ROOT.parents]:
    if (c / "app" / "src").exists():
        ROOT = c; break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "app" / "validation" / "03_alias_suggestion"))

# Nạp .env (giống UI) để LLMConfig thấy OPENAI_API_KEY
env = ROOT / ".env"
if env.exists():
    for line in env.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("="); os.environ.setdefault(k.strip(), v.strip())

import importlib
import config_reconstruction as R
importlib.reload(R)
from app.src.llm.config import LLMConfig

VIRUS = "FMDV"
MIN_IOU = 0.90
PROVIDER = None if LLMConfig.from_env().available else R._MockLLMProvider()
print("LLM:", "REAL" if PROVIDER is None else "MOCK (no API key)")

LLM: REAL


## Chạy — seed → suggest → LLM → config_temp → so per-canonical

Ghi ra `outputs/config_temp_FMDV.json`, `reconstruction_FMDV.tsv` (per-canonical), `..._detail.tsv` (từng ca sai).

In [2]:
per_canon, detail = R.run_virus(VIRUS, R.DATASETS[VIRUS], MIN_IOU, llm_provider=PROVIDER)
R.summarize(VIRUS, per_canon)


===== FMDV =====
  PRECISION (config_temp → truth): 41/41 = 100.0%
     wrong_gene 0  false_save 0  not_in_truth 0
  RECALL (truth ∩ corpus → config_temp): 35/35 = 100.0%
     missed 0  (parked, not scored: 31)


## Bảng per-canonical (đọc từng gene)

In [3]:
pd.set_option("display.width", 200)
per_canon

,canonical,temp_saved,precision_correct,wrong_gene,false_save,not_in_truth,precision_pct,recall_denom,recall_found,missed,recall_pct,parked
0,2A,2,2,0,0,0,100.0,2,2,0,100.0,0
1,2B,2,2,0,0,0,100.0,2,2,0,100.0,0
2,2C,2,2,0,0,0,100.0,2,2,0,100.0,0
3,3A,2,2,0,0,0,100.0,2,2,0,100.0,0
4,3B,6,6,0,0,0,100.0,6,6,0,100.0,1
5,3B1,0,0,0,0,0,NaN,0,0,0,NaN,2
6,3B2,0,0,0,0,0,NaN,0,0,0,NaN,2
7,3B3,0,0,0,0,0,NaN,0,0,0,NaN,2
8,3Cpro,5,5,0,0,0,100.0,3,3,0,100.0,4
9,3Dpol,5,5,0,0,0,100.0,3,3,0,100.0,4


## Precision — chỗ tool lưu SAI

`wrong_gene` (map nhầm gene) và `false_save` (lưu nhầm rác) là lỗi đắt, lý tưởng = 0.
`not_in_truth` = tool gợi ý mà truth im lặng → **kiểm truth có thiếu không**.

In [4]:
prec_issues = detail[detail["side"]=="precision"]
prec_issues if len(prec_issues) else "— precision sạch, không lưu sai gì —"

'— precision sạch, không lưu sai gì —'

## Recall — chỗ tool BỎ SÓT (đây là chỗ bắt bug kiểu sM)

`missed(temp→...)` cho biết alias thật bị đẩy đi đâu: `excluded` = bị loại nhầm (nguy hiểm nhất),
một canon khác = map lệch.

In [5]:
miss = detail[(detail["side"]=="recall") & (detail["issue"].str.startswith("missed"))]
miss if len(miss) else "— recall đầy đủ, không bỏ sót alias nào có trong corpus —"

'— recall đầy đủ, không bỏ sót alias nào có trong corpus —'

## Parked — alias truth KHÔNG có trong corpus (không tính điểm)

Đây là alias curated từ literature/kinh nghiệm mà lô 100 record này không dùng. Tool không thể gợi ý được,
nên loại khỏi mẫu số recall. Liệt kê để minh bạch.

In [6]:
parked = detail[detail["issue"]=="parked_not_in_corpus"]
print(f"{len(parked)} alias park (không tính accuracy)")
parked[["canonical","name"]]

25 alias park (không tính accuracy)


,canonical,name
0,3B,vpg
7,3Cpro,3cpro
8,3Cpro,protease3c
9,3Cpro,3cprotease
10,3Cpro,propeptide
11,3Dpol,3dpol
12,3Dpol,rnadependentrnapolymerase
13,3Dpol,3dpolymerase
14,3Dpol,polpeptide
15,Lpro,lpro


## Ghi chú cho paper

- **Precision** = độ tin của cái tool tự lưu; **Recall** (gate corpus) = độ phủ trên phần tool *có thể*
  gợi ý. Hai số tách riêng, không gộp.
- **Gate corpus** là bắt buộc để công bằng: alias literature-only không phải lỗi lifting.
- **Recall detail là nơi lộ bug kiểu sM** — alias thật có trong corpus mà bị `missed(temp→excluded)`.